# 13 — Dijkstra's Algorithm on an Occupancy Grid

**Section:** Motion Planning · **Mirrors MATLAB:** *Motion Planners (RRT, PRM, Hybrid A\*)* — Dijkstra baseline

Dijkstra is A\* with a zero heuristic: it explores cells in pure cost-to-come order. This guarantees optimality but expands more nodes than A\* (which uses the goal-distance heuristic to bias exploration).

Comparing the two on the same map helps illustrate why an admissible heuristic matters.

## Intuition — what's actually going on?

Dijkstra's algorithm is A\* without the "head toward the goal" hint. It explores the grid as an expanding circle of equal-cost cells around the start, like a ripple in water. It's guaranteed to find the shortest path, but it doesn't know where the goal is, so it explores *everywhere* equally until it bumps into the goal.

A\* is strictly faster on the same map because the heuristic `h(n)` (here Euclidean distance to goal) biases the search to prefer cells that look closer to the goal. The picture from this notebook shows Dijkstra exploring **1190** cells vs A\* exploring **522** for the same path — A\* expanded less than half as many nodes.

When would you use plain Dijkstra anyway? When you have *no* useful heuristic — e.g. you're computing shortest paths in an abstract graph where there's no spatial structure.

### Compatibility check — math ↔ code

| Math | Code |
|---|---|
| $f(n) = g(n) + 0 = g(n)$ (zero heuristic) | priority queue uses `(d, current, parent)` with no $h$ term |
| Edge cost = step distance | `nd = d + np.hypot(dy, dx)` |
| Always pop smallest $g$ | `heapq.heappop(heap)` |
| Relax neighbor: $g(n') = \min(g(n'),\ g(n) + c(n,n'))$ | `if nd < dist.get((ny, nx), np.inf): dist[(ny,nx)] = nd; heapq.heappush(...)` |
| Skip closed nodes | `if cur in came: continue` |
| Path reconstruction | `while cur is not None: path.append(cur); cur = came[cur]` |

Notice that the *only* difference from notebook 01 is the absence of the $h$ term in the priority — Dijkstra is the special case A\* with $h \equiv 0$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import heapq

np.random.seed(42)

H, W = 30, 50
grid = np.zeros((H, W), dtype=np.uint8)
for _ in range(40):
    y, x = np.random.randint(2, H - 3), np.random.randint(2, W - 3)
    grid[y - 1:y + 2, x - 1:x + 2] = 1
start, goal = (2, 2), (H - 3, W - 3)
grid[start] = 0; grid[goal] = 0


In [ ]:
def dijkstra(grid, start, goal):
    H, W = grid.shape
    heap = [(0.0, start, None)]
    came_from, dist = {}, {start: 0.0}
    nbrs = [(-1, 0), (1, 0), (0, -1), (0, 1),
            (-1, -1), (-1, 1), (1, -1), (1, 1)]
    while heap:
        d, current, parent = heapq.heappop(heap)
        if current in came_from:
            continue
        came_from[current] = parent
        if current == goal:
            path = []
            while current is not None:
                path.append(current); current = came_from[current]
            return path[::-1], dist
        for dy, dx in nbrs:
            ny, nx = current[0] + dy, current[1] + dx
            if 0 <= ny < H and 0 <= nx < W and grid[ny, nx] == 0:
                nd = d + np.hypot(dy, dx)
                if nd < dist.get((ny, nx), np.inf):
                    dist[(ny, nx)] = nd
                    heapq.heappush(heap, (nd, (ny, nx), current))
    return None, dist


path, dist = dijkstra(grid, start, goal)
print(f"Path: {len(path)} cells   expanded: {len(dist)} cells")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(grid, cmap='Greys', origin='lower')
expanded = np.full_like(grid, np.nan, dtype=float)
for (y, x), c in dist.items():
    expanded[y, x] = c
ax.imshow(expanded, cmap='viridis', alpha=0.4, origin='lower')

ys, xs = zip(*path)
ax.plot(xs, ys, 'r-', lw=2.5, label='Dijkstra path')
ax.plot(start[1], start[0], 'go', ms=14, label='Start')
ax.plot(goal[1], goal[0], 'b*', ms=18, label='Goal')
ax.legend(); ax.set_title("Dijkstra — expanded cells shaded by g-score (uses zero heuristic)")
plt.tight_layout()
plt.show()


## References & rigor notes

**Corollary** (from A\\* with $h \equiv 0$). *Dijkstra returns the shortest path from $s$ to $t$ on any graph with non-negative edge weights.*

**Complexity** by data structure:
- Binary heap: $O((|V| + |E|) \log |V|)$
- Fibonacci heap: $O(|V| \log |V| + |E|)$
- Adjacency-matrix without priority queue: $O(|V|^2)$

**Fails on negative weights.** Choose by graph properties:
- **Dijkstra** ($O((|V|+|E|)\log|V|)$): non-negative weights, single-source.
- **Bellman-Ford** ($O(|V| \cdot |E|)$): handles negative weights without negative cycles, single-source.
- **Johnson's** ($O(|V|^2 \log |V| + |V| \cdot |E|)$): all-pairs after reweighting.

**DP framing.** Equivalently, Dijkstra is dynamic programming applied to the cost-to-go value function $V^*(n) = \min_\pi c(\pi)$, expanding nodes in order of $V^*$ — the canonical instance of Bellman's label-correcting framework.

**References.**
- Dijkstra, E. W. (1959). *A note on two problems in connexion with graphs*. Numerische Mathematik, 1(1), 269-271.
- Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2009). *Introduction to Algorithms*, 3rd ed., MIT Press, ch. 24.
